#### imports

In [26]:
import utils as ut
import pandas as pd
import os, sys
import numpy as np
import networkx as nx
import numpy as np

In [4]:
pd.__version__

'2.0.3'

In [5]:
data = ut.read_data("F:\TFG\datasets\\raw_datasets\datalake.csv")

f:\TFG\code\old_stuff\experiments\utils.py:20: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path,sep=';',decimal=',',parse_dates=['Date'],date_format="%d/%m/%Y")


In [6]:
data

,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,AHW,B365H,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA
0,0,8,19,D1,0,2000-08-11,T00-01,Dortmund,Hansa Rostock,1,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,20,D1,1,2000-08-12,T00-01,Bayern Munich,Hertha,4,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,15,35,D1,2,2000-08-12,T00-01,Freiburg,Stuttgart,4,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,17,29,D1,3,2000-08-12,T00-01,Hamburg,Munich 1860,2,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,23,4,D1,4,2000-08-12,T00-01,Kaiserslautern,Bochum,0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,58541,258,243,SP2,10828,2000-06-04,T99-00,Extremadura,Badajoz,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58542,58542,271,274,SP2,10829,2000-06-04,T99-00,Las Palmas,Levante,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58543,58543,277,314,SP2,10830,2000-06-04,T99-00,Logrones,Villarreal,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58544,58544,288,297,SP2,10831,2000-06-04,T99-00,Osasuna,Recreativo,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
data.columns

Index(['matchId', 'id_H', 'id_A', 'Div', 'div_order', 'Date', 'season',
       'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'Attendance', 'Referee',
       'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR',
       'AR', 'HO', 'AO', 'HHW', 'AHW', 'B365H', 'B365D', 'B365A', 'MaxH',
       'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA'],
      dtype='object')

In [8]:
COLS_META   = ['matchId','Div','Date','season','id_H','id_A','HomeTeam','AwayTeam']
data[COLS_META]

,matchId,Div,Date,season,id_H,id_A,HomeTeam,AwayTeam
0,0,D1,2000-08-11,T00-01,8,19,Dortmund,Hansa Rostock
1,1,D1,2000-08-12,T00-01,2,20,Bayern Munich,Hertha
2,2,D1,2000-08-12,T00-01,15,35,Freiburg,Stuttgart
3,3,D1,2000-08-12,T00-01,17,29,Hamburg,Munich 1860
4,4,D1,2000-08-12,T00-01,23,4,Kaiserslautern,Bochum
...,...,...,...,...,...,...,...,...
58541,58541,SP2,2000-06-04,T99-00,258,243,Extremadura,Badajoz
58542,58542,SP2,2000-06-04,T99-00,271,274,Las Palmas,Levante
58543,58543,SP2,2000-06-04,T99-00,277,314,Logrones,Villarreal
58544,58544,SP2,2000-06-04,T99-00,288,297,Osasuna,Recreativo


In [28]:
import torch

In [46]:
def rank_probability_score(logits,actual):
    rps = torch.sum(np.power(logits-actual,2),1)
    rps = rps / logits.shape[1]
    return rps

def avg_rps(logits,actual):
    res = rank_probability_score(logits,actual)
    return float(torch.mean(res,axis=0))

In [47]:
rps = avg_rps(logits,actual)
rps, type(rps)

(0.18779999017715454, float)

# PI RATING

## FUNCTIONS

In [61]:
def get_rate_global(elemH,elemA):
    return (elemH+elemA)/2

def asign_col(df,col_name,value):
    df.loc[:,col_name] = value

## LOGIC

In [62]:
data_initialization = data[data.Date<"1996-07-01"]
old_data_init = data_initialization.copy()
old_data_init

,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,AHW,B365H,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA
6280,6280,2,15,D1,6280,1993-07-08,T93-94,Bayern Munich,Freiburg,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6281,6281,8,24,D1,6281,1993-07-08,T93-94,Dortmund,Karlsruhe,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6282,6282,10,26,D1,6282,1993-07-08,T93-94,Duisburg,Leverkusen,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6283,6283,13,23,D1,6283,1993-07-08,T93-94,FC Koln,Kaiserslautern,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6284,6284,17,30,D1,6284,1993-07-08,T93-94,Hamburg,Nurnberg,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56905,56905,254,242,SP2,9192,1996-01-12,T96-97,Ecija,Ath Madrid B,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56906,56906,271,243,SP2,9193,1996-01-12,T96-97,Las Palmas,Badajoz,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56907,56907,272,255,SP2,9194,1996-01-12,T96-97,Leganes,Eibar,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56908,56908,284,274,SP2,9195,1996-01-12,T96-97,Merida,Levante,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [64]:
data_initialization = old_data_init.copy()

cols_pirating = ["matchId","id_H","id_A","HomeTeam","AwayTeam","Div","Date","FTHG","FTAG","FTR"] 
data_initialization = data_initialization[cols_pirating]

asign_col(data_initialization,"FTHG",data_initialization.FTHG.astype(float))
asign_col(data_initialization,"FTAG",data_initialization.FTAG.astype(float))

In [65]:
asign_col(data_initialization,"rate_home",0.0)
asign_col(data_initialization,"rate_away",0.0)
asign_col(data_initialization,"new_rate_home",0.0)
asign_col(data_initialization,"new_rate_away",0.0)
rate_global = get_rate_global(data_initialization.rate_home,data_initialization.rate_away)
asign_col(data_initialization,"rate_global",rate_global)
asign_col(data_initialization,"new_rate_global",0.0)

In [66]:
teams = data.HomeTeam.unique()
print(teams.shape)
dict_rates = { t:{"rate_home":0,"rate_away":0,"rate_global":0} for t in teams }

(275,)


In [67]:
data_initialization

,matchId,id_H,id_A,HomeTeam,AwayTeam,Div,Date,FTHG,FTAG,FTR,rate_home,rate_away,new_rate_home,new_rate_away,rate_global,new_rate_global
6280,6280,2,15,Bayern Munich,Freiburg,D1,1993-07-08,3.0,1.0,H,0.0,0.0,0.0,0.0,0.0,0.0
6281,6281,8,24,Dortmund,Karlsruhe,D1,1993-07-08,2.0,1.0,H,0.0,0.0,0.0,0.0,0.0,0.0
6282,6282,10,26,Duisburg,Leverkusen,D1,1993-07-08,2.0,2.0,D,0.0,0.0,0.0,0.0,0.0,0.0
6283,6283,13,23,FC Koln,Kaiserslautern,D1,1993-07-08,0.0,2.0,A,0.0,0.0,0.0,0.0,0.0,0.0
6284,6284,17,30,Hamburg,Nurnberg,D1,1993-07-08,5.0,2.0,H,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56905,56905,254,242,Ecija,Ath Madrid B,SP2,1996-01-12,0.0,1.0,A,0.0,0.0,0.0,0.0,0.0,0.0
56906,56906,271,243,Las Palmas,Badajoz,SP2,1996-01-12,2.0,2.0,D,0.0,0.0,0.0,0.0,0.0,0.0
56907,56907,272,255,Leganes,Eibar,SP2,1996-01-12,1.0,2.0,A,0.0,0.0,0.0,0.0,0.0,0.0
56908,56908,284,274,Merida,Levante,SP2,1996-01-12,1.0,1.0,D,0.0,0.0,0.0,0.0,0.0,0.0


In [68]:
asign_col(data_initialization,"gd_actual", data_initialization.FTHG - data_initialization.FTAG)
asign_col(data_initialization,"gd_pred",0.0)
asign_col(data_initialization,"error",0.0)

In [69]:
data_initialization

,matchId,id_H,id_A,HomeTeam,AwayTeam,Div,Date,FTHG,FTAG,FTR,rate_home,rate_away,new_rate_home,new_rate_away,rate_global,new_rate_global,gd_actual,gd_pred,error
6280,6280,2,15,Bayern Munich,Freiburg,D1,1993-07-08,3.0,1.0,H,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
6281,6281,8,24,Dortmund,Karlsruhe,D1,1993-07-08,2.0,1.0,H,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6282,6282,10,26,Duisburg,Leverkusen,D1,1993-07-08,2.0,2.0,D,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6283,6283,13,23,FC Koln,Kaiserslautern,D1,1993-07-08,0.0,2.0,A,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,0.0,0.0
6284,6284,17,30,Hamburg,Nurnberg,D1,1993-07-08,5.0,2.0,H,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56905,56905,254,242,Ecija,Ath Madrid B,SP2,1996-01-12,0.0,1.0,A,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0
56906,56906,271,243,Las Palmas,Badajoz,SP2,1996-01-12,2.0,2.0,D,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
56907,56907,272,255,Leganes,Eibar,SP2,1996-01-12,1.0,2.0,A,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0
56908,56908,284,274,Merida,Levante,SP2,1996-01-12,1.0,1.0,D,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [70]:
b = 10
c = 3
lamda = .01
gamma = 0.2

def error_func(actual,pred):
        error = abs(actual-pred)
        error = c * np.log10(1+error)
        error_H = -error
        error_A = -error
        if actual>pred:
                error_H = error
        if actual<pred: 
                error_A = error
        return error_H, error_A

In [72]:

for row in data_initialization.itertuples():
    idx = row.Index
    team_H = row.HomeTeam
    team_A = row.AwayTeam
    data_initialization.loc[idx,"rate_home"] = dict_rates[team_H]["rate_home"]
    data_initialization.loc[idx,"rate_away"] = dict_rates[team_A]["rate_away"]
    gd_H_pred = b ** (abs(dict_rates[team_H]["rate_home"])/c) - 1
    gd_A_pred = b ** (abs(dict_rates[team_A]["rate_away"])/c) - 1
    data_initialization.loc[idx,"gd_pred"] = gd_H_pred - gd_A_pred
    error_H, error_A = error_func(row.gd_actual,row.gd_pred)
    data_initialization.loc[idx,"error"] = abs(error_H)

    new_rate_home_H = dict_rates[team_H]["rate_home"] + error_H * lamda
    new_rate_home_A = dict_rates[team_H]["rate_away"] + (new_rate_home_H - dict_rates[team_H]["rate_home"]) * gamma
    new_rate_away_A = dict_rates[team_A]["rate_away"] + error_A * lamda
    new_rate_away_H = dict_rates[team_A]["rate_home"] + (new_rate_away_A - dict_rates[team_A]["rate_away"]) * gamma

    dict_rates[team_H]["rate_home"] = new_rate_home_H
    dict_rates[team_A]["rate_home"] = new_rate_away_H
    dict_rates[team_A]["rate_global"] = (new_rate_home_H + new_rate_away_H) / 2
    dict_rates[team_A]["rate_away"] = new_rate_away_A
    dict_rates[team_A]["rate_home"] = new_rate_away_H
    dict_rates[team_A]["rate_global"] = (new_rate_away_H + new_rate_away_A) / 2
        
    data_initialization.loc[idx,"new_rate_home"] = new_rate_home_H
    data_initialization.loc[idx,"new_rate_away"] = new_rate_away_A
    

In [73]:
data_initialization

,matchId,id_H,id_A,HomeTeam,AwayTeam,Div,Date,FTHG,FTAG,FTR,rate_home,rate_away,new_rate_home,new_rate_away,rate_global,new_rate_global,gd_actual,gd_pred,error
6280,6280,2,15,Bayern Munich,Freiburg,D1,1993-07-08,3.0,1.0,H,0.000000,0.000000,0.014314,-0.014314,0.0,0.0,2.0,0.000000,1.431364
6281,6281,8,24,Dortmund,Karlsruhe,D1,1993-07-08,2.0,1.0,H,0.000000,0.000000,0.009031,-0.009031,0.0,0.0,1.0,0.000000,0.903090
6282,6282,10,26,Duisburg,Leverkusen,D1,1993-07-08,2.0,2.0,D,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000
6283,6283,13,23,FC Koln,Kaiserslautern,D1,1993-07-08,0.0,2.0,A,0.000000,0.000000,-0.014314,0.014314,0.0,0.0,-2.0,0.000000,1.431364
6284,6284,17,30,Hamburg,Nurnberg,D1,1993-07-08,5.0,2.0,H,0.000000,0.000000,0.018062,-0.018062,0.0,0.0,3.0,0.000000,1.806180
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56905,56905,254,242,Ecija,Ath Madrid B,SP2,1996-01-12,0.0,1.0,A,0.010120,-0.018062,0.001089,-0.009031,0.0,0.0,-1.0,-0.006162,0.903090
56906,56906,271,243,Las Palmas,Badajoz,SP2,1996-01-12,2.0,2.0,D,0.009031,0.014314,0.009031,0.014314,0.0,0.0,0.0,-0.004091,0.000000
56907,56907,272,255,Leganes,Eibar,SP2,1996-01-12,1.0,2.0,A,0.000000,0.000000,-0.009031,0.009031,0.0,0.0,-1.0,0.000000,0.903090
56908,56908,284,274,Merida,Levante,SP2,1996-01-12,1.0,1.0,D,0.007131,-0.009031,0.007131,-0.009031,0.0,0.0,0.0,-0.001468,0.000000


In [76]:
final_rates = pd.DataFrame(dict_rates).T.reset_index()
teams = data[['HomeTeam','AwayTeam']].drop_duplicates()
final_rates.merge(teams, left_on='index', right_on='HomeTeam', how='left').sort_values("rate_global",ascending=False)

,index,rate_home,rate_away,rate_global,HomeTeam,AwayTeam
1654,Man United,0.611902,0.247909,0.434421,Man United,Hull
1645,Man United,0.611902,0.247909,0.434421,Man United,Birmingham
1647,Man United,0.611902,0.247909,0.434421,Man United,Portsmouth
1649,Man United,0.611902,0.247909,0.434421,Man United,Norwich
1650,Man United,0.611902,0.247909,0.434421,Man United,Crystal Palace
...,...,...,...,...,...,...
1770,Ipswich,-0.215195,-0.334690,-0.274943,Ipswich,Crystal Palace
1769,Ipswich,-0.215195,-0.334690,-0.274943,Ipswich,Nott'm Forest
1768,Ipswich,-0.215195,-0.334690,-0.274943,Ipswich,Swindon
1767,Ipswich,-0.215195,-0.334690,-0.274943,Ipswich,QPR


# PAGE RANK

In [23]:
old_data = data.copy()
data = data[data.Div=='SP1']
ut.getPoints(data,"FTHG","FTAG","Points_H")
ut.getPoints(data,"FTAG","FTHG","Points_A")
data

,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Points_H,Points_A
37021,37021,190,212,SP1,0,2000-09-09,T00-01,Barcelona,Malaga,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
37022,37022,206,188,SP1,1,2000-09-09,T00-01,La Coruna,Ath Bilbao,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
37023,37023,219,227,SP1,2,2000-09-09,T00-01,Real Madrid,Valencia,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
37024,37024,224,222,SP1,3,2000-09-09,T00-01,Sociedad,Santander,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
37025,37025,232,198,SP1,4,2000-09-09,T00-01,Zaragoza,Espanol,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47708,47708,223,229,SP1,10687,2000-05-19,T99-00,Sevilla,Vallecano,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3
47709,47709,227,232,SP1,10688,2000-05-19,T99-00,Valencia,Zaragoza,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
47710,47710,212,222,SP1,10689,2000-05-20,T99-00,Malaga,Santander,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
47711,47711,216,191,SP1,10690,2000-05-20,T99-00,Numancia,Betis,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3


In [9]:
def get_derby_id(data,columns):
    unique_ids = np.sort(data[columns].values,axis=1)
    df_derby = pd.DataFrame(np.unique(unique_ids,axis=0),columns=["id1","id2"])
    df_derby.reset_index(inplace=True,names='derby')
    return df_derby, unique_ids

def add_column_from_df(df,other_df,list_values,columns):
    for i,c in enumerate(columns):
        df.loc[:,c] = list_values[:,i]
    df = df.merge(other_df,on=["id1","id2"])
    return df

In [282]:
match_sorted_list, match_sorted = get_derby_id(data,["id_H","id_A"])
data = add_column_from_df(data,match_sorted_list,match_sorted,["id1","id2"])
data

<ipython-input-281-08fd3addd4b9>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,c] = list_values[:,i]
<ipython-input-281-08fd3addd4b9>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,c] = list_values[:,i]


,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,MaxD,MaxA,AvgH,AvgD,AvgA,Points_H,Points_A,id1,id2,derby
0,37021,190,212,SP1,0,2000-09-09,T00-01,Barcelona,Malaga,2,...,NaN,NaN,NaN,NaN,NaN,3,0,190,212,206
1,37213,212,190,SP1,192,2001-01-27,T00-01,Malaga,Barcelona,0,...,NaN,NaN,NaN,NaN,NaN,1,1,190,212,206
2,37484,212,190,SP1,463,2001-10-20,T01-02,Malaga,Barcelona,1,...,NaN,NaN,NaN,NaN,NaN,1,1,190,212,206
3,37676,190,212,SP1,655,2002-03-03,T01-02,Barcelona,Malaga,5,...,NaN,NaN,NaN,NaN,NaN,3,0,190,212,206
4,37947,212,190,SP1,926,2003-12-01,T02-03,Malaga,Barcelona,0,...,NaN,NaN,NaN,NaN,NaN,1,1,190,212,206
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,47290,221,185,SP1,10269,1999-05-16,T98-99,Salamanca,Alaves,1,...,NaN,NaN,NaN,NaN,NaN,3,0,185,221,25
10688,47107,199,213,SP1,10086,1999-03-01,T98-99,Extremadura,Mallorca,1,...,NaN,NaN,NaN,NaN,NaN,3,0,199,213,428
10689,47297,213,199,SP1,10276,1999-05-23,T98-99,Mallorca,Extremadura,2,...,NaN,NaN,NaN,NaN,NaN,3,0,199,213,428
10690,47142,230,199,SP1,10121,1999-01-24,T98-99,Villarreal,Extremadura,1,...,NaN,NaN,NaN,NaN,NaN,1,1,199,230,440


In [283]:
points_H = data.melt(id_vars=["derby","id_H","Date"],value_vars=["Points_H"])
points_A = data.melt(id_vars=["derby","id_A","Date"],value_vars=["Points_A"])
points_H.columns = points_A.columns = ["derby","id","Date","variable","value"]
points_df = pd.concat([points_H,points_A]) 
points_df

,derby,id,Date,variable,value
0,206,190,2000-09-09,Points_H,3
1,206,212,2001-01-27,Points_H,1
2,206,212,2001-10-20,Points_H,1
3,206,190,2002-03-03,Points_H,3
4,206,212,2003-12-01,Points_H,1
...,...,...,...,...,...
10687,25,185,1999-05-16,Points_A,0
10688,428,213,1999-03-01,Points_A,0
10689,428,199,1999-05-23,Points_A,0
10690,440,199,1999-01-24,Points_A,1


In [284]:
page_rank_matrix = ut.compute_lag(points_df,lag=5,cols_group=["derby","id"],col_window="Date",min_samples=2,aggregations={"value":"mean"})
page_rank_matrix = page_rank_matrix.reset_index().sort_values("Date").dropna()
page_rank_matrix

,derby,id,Date,value_5
19333,769,225,1993-03-10,1.000000
14421,557,224,1993-03-10,1.600000
4763,171,219,1993-03-10,1.800000
9770,317,193,1993-03-10,1.800000
1159,62,186,1993-03-10,2.000000
...,...,...,...,...
19870,780,227,2021-12-05,1.200000
19796,780,223,2021-12-05,1.800000
2351,114,188,2021-12-05,2.333333
5042,176,224,2021-12-05,0.200000


In [10]:
def get_between(df,date1,date2):
    mask_date1 = df.Date>date1
    mask_date2 = df.Date<=date2
    return df[mask_date1 & mask_date2]

In [286]:
# idxs = page_rank_matrix[page_rank_matrix.Date<="2000-04-06"]
df_filtered = get_between(page_rank_matrix,"2008-07-01","2011-07-01")
idxs = df_filtered[["derby","id"]].drop_duplicates(keep='last').index
idxs

Index([ 2697, 17223,  7908, 17284,  1502, 17163,  8346,  4568,  6301, 15862,
       ...
       20690, 20381,  4171,  1857, 20698, 12183,  4155, 17400, 17418, 12123],
      dtype='int64', length=540)

In [287]:
page_rank_matrix_filt = page_rank_matrix.loc[idxs].drop(columns='Date')
# page_rank_matrix_filt = page_rank_matrix_filt.set_index("derby").sort_index()
page_rank_matrix_filt = page_rank_matrix_filt.sort_values("derby")
page_rank_matrix_filt

,derby,id,value_5
1225,66,188,2.000000
1207,66,187,1.333333
1243,67,189,2.333333
1237,67,187,0.800000
1255,68,187,0.666667
...,...,...,...
21170,812,228,2.600000
21220,814,228,2.250000
21259,814,232,0.800000
21344,818,230,2.600000


In [288]:
page_rank_matrix_filt_1 = page_rank_matrix_filt[::2]
page_rank_matrix_filt_2 = page_rank_matrix_filt[1::2]
page_rank_matrix_pivoted = page_rank_matrix_filt_1.merge(page_rank_matrix_filt_2, on='derby', suffixes=('_1','_2'))
page_rank_matrix_pivoted

,derby,id_1,value_5_1,id_2,value_5_2
0,66,188,2.000000,187,1.333333
1,67,189,2.333333,187,0.800000
2,68,187,0.666667,190,2.200000
3,69,191,1.750000,187,1.750000
4,74,187,1.000000,198,1.800000
...,...,...,...,...,...
259,808,230,2.400000,227,0.600000
260,810,232,0.200000,227,2.600000
261,812,230,0.200000,228,2.600000
262,814,228,2.250000,232,0.800000


In [52]:
team_id_name = data[["id_H","HomeTeam"]].set_index("id_H")["HomeTeam"].to_dict()

In [53]:
page_rank_matrix_pivoted.loc[:,"HomeTeam"] = page_rank_matrix_pivoted.id_1.map(team_id_name)
page_rank_matrix_pivoted.loc[:,"AwayTeam"] = page_rank_matrix_pivoted.id_2.map(team_id_name)

In [54]:
page_rank_matrix_pivoted.HomeTeam.unique().shape

(26,)

In [62]:
teams = page_rank_matrix_pivoted.HomeTeam.unique()
matrix_page_rank = pd.DataFrame(columns=teams,index=teams)

for row in page_rank_matrix_pivoted.itertuples():
    matrix_page_rank.loc[row.HomeTeam,row.AwayTeam] = row.value_5_1
    matrix_page_rank.loc[row.AwayTeam,row.HomeTeam] = row.value_5_2

In [63]:
dict_team_page = { t:i for t,i in enumerate(matrix_page_rank.iloc[0].index)}
dict_team_page

{0: 'Ath Bilbao',
 1: 'Ath Madrid',
 2: 'Almeria',
 3: 'Betis',
 4: 'Getafe',
 5: 'La Coruna',
 6: 'Levante',
 7: 'Malaga',
 8: 'Mallorca',
 9: 'Osasuna',
 10: 'Sevilla',
 11: 'Sp Gijon',
 12: 'Valencia',
 13: 'Valladolid',
 14: 'Hercules',
 15: 'Numancia',
 16: 'Recreativo',
 17: 'Tenerife',
 18: 'Espanol',
 19: 'Santander',
 20: 'Zaragoza',
 21: 'Barcelona',
 22: 'Real Madrid',
 23: 'Sociedad',
 24: 'Villarreal',
 25: 'Vallecano'}

In [64]:
# matrix_page_rank = matrix_page_rank.dropna().drop(columns='NaN')
matrix_page_rank = matrix_page_rank.sort_index()
matrix_page_rank = matrix_page_rank[sorted(matrix_page_rank.columns)]
matrix_page_rank

,Almeria,Ath Bilbao,Ath Madrid,Barcelona,Betis,Espanol,Getafe,Hercules,La Coruna,Levante,...,Santander,Sevilla,Sociedad,Sp Gijon,Tenerife,Valencia,Valladolid,Vallecano,Villarreal,Zaragoza
Almeria,NaN,1.333333,0.8,0.666667,1.75,1.0,2.0,NaN,1.333333,0.8,...,0.666667,1.0,1.666667,1.4,NaN,0.333333,1.4,NaN,0.6,1.0
Ath Bilbao,2.0,NaN,1.2,0.4,1.6,0.2,0.6,3.0,1.0,2.0,...,1.4,0.6,2.6,1.5,3.0,1.8,1.4,NaN,0.8,1.2
Ath Madrid,2.333333,1.8,NaN,0.8,1.8,1.4,1.6,NaN,1.2,3.0,...,2.2,1.8,0.0,2.6,2.0,0.4,0.8,NaN,1.4,1.0
Barcelona,2.2,2.2,2.0,NaN,1.0,1.6,2.2,0.0,3.0,3.0,...,2.6,1.4,1.6,2.6,2.2,1.0,2.2,NaN,1.8,2.6
Betis,1.75,1.0,1.2,1.6,NaN,1.6,2.0,NaN,1.2,1.5,...,1.6,0.2,NaN,1.4,NaN,0.8,2.0,NaN,0.8,NaN
Espanol,1.8,2.6,1.4,1.0,1.0,NaN,1.6,NaN,1.8,1.6,...,0.8,1.8,1.6,1.0,2.6,0.6,1.0,NaN,1.2,1.0
Getafe,1.4,1.8,1.0,0.4,1.8,1.0,NaN,NaN,0.8,1.6,...,1.4,1.8,1.0,2.0,NaN,2.0,1.5,NaN,0.2,1.6
Hercules,NaN,1.5,NaN,1.5,NaN,NaN,NaN,NaN,1.5,NaN,...,NaN,3.0,3.0,1.0,NaN,0.0,NaN,NaN,NaN,2.0
La Coruna,0.8,1.6,1.8,0.0,1.2,1.2,2.0,3.0,NaN,2.0,...,1.6,1.4,1.6,0.5,1.6,0.4,2.6,NaN,1.8,0.8
Levante,2.4,1.333333,0.4,0.2,2.4,0.666667,1.666667,NaN,0.8,NaN,...,2.333333,0.2,1.0,1.666667,NaN,0.8,NaN,NaN,0.8,2.0


In [58]:
data[data.id_H==222]

,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,MaxD,MaxA,AvgH,AvgD,AvgA,Points_H,Points_A,id1,id2,derby
135,37218,222,224,SP1,197,2001-01-28,T00-01,Santander,Sociedad,1,...,NaN,NaN,NaN,NaN,NaN,0,3,222,224,768
137,38032,222,224,SP1,1011,2003-03-15,T02-03,Santander,Sociedad,1,...,NaN,NaN,NaN,NaN,NaN,0,3,222,224,768
138,38183,222,224,SP1,1162,2003-09-13,T03-04,Santander,Sociedad,0,...,NaN,NaN,NaN,NaN,NaN,0,3,222,224,768
141,38946,222,224,SP1,1925,2005-04-12,T05-06,Santander,Sociedad,2,...,NaN,NaN,NaN,NaN,NaN,1,1,222,224,768
143,39366,222,224,SP1,2345,2007-01-14,T06-07,Santander,Sociedad,1,...,NaN,NaN,NaN,NaN,NaN,3,0,222,224,768
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10556,45841,222,221,SP1,8820,1995-12-20,T95-96,Santander,Salamanca,2.0,...,NaN,NaN,NaN,NaN,NaN,3,0,221,222,757
10558,46580,222,221,SP1,9559,1997-08-31,T97-98,Santander,Salamanca,1,...,NaN,NaN,NaN,NaN,NaN,3,0,221,222,757
10560,47049,222,221,SP1,10028,1998-11-15,T98-99,Santander,Salamanca,4,...,NaN,NaN,NaN,NaN,NaN,3,0,221,222,757
10603,46401,222,199,SP1,9380,1997-02-03,T96-97,Santander,Extremadura,2.0,...,NaN,NaN,NaN,NaN,NaN,0,3,199,222,432


In [67]:
page_rank_matrix_pivoted

,derby,id_1,value_5_1,id_2,value_5_2,HomeTeam,AwayTeam
0,66,188,2.000000,187,1.333333,Ath Bilbao,Almeria
1,67,189,2.333333,187,0.800000,Ath Madrid,Almeria
2,68,187,0.666667,190,2.200000,Almeria,Barcelona
3,69,191,1.750000,187,1.750000,Betis,Almeria
4,74,187,1.000000,198,1.800000,Almeria,Espanol
...,...,...,...,...,...,...,...
259,808,230,2.400000,227,0.600000,Villarreal,Valencia
260,810,232,0.200000,227,2.600000,Zaragoza,Valencia
261,812,230,0.200000,228,2.600000,Villarreal,Valladolid
262,814,228,2.250000,232,0.800000,Valladolid,Zaragoza


In [81]:
import pandas as pd
import networkx as nx

# Create an empty directed graph
graph = nx.DiGraph()

# Iterate over the DataFrame rows
for _, row in page_rank_matrix_pivoted.iterrows():
    source = row['HomeTeam']
    target = row['AwayTeam']
    weight = row['value_5_2']
    
    # Add edges only for non-NaN values
    if not pd.isna(weight):
        graph.add_edge(source, target, weight=weight)

    source = row['AwayTeam']
    target = row['HomeTeam']
    weight = row['value_5_1']
    
    # Add edges only for non-NaN values
    if not pd.isna(weight):
        graph.add_edge(source, target, weight=weight)

# Print the graph
print(graph.edges(data=True))


[('Ath Bilbao', 'Almeria', {'weight': 1.3333333333333333}), ('Ath Bilbao', 'Ath Madrid', {'weight': 1.8}), ('Ath Bilbao', 'Barcelona', {'weight': 2.2}), ('Ath Bilbao', 'Betis', {'weight': 1.0}), ('Ath Bilbao', 'Espanol', {'weight': 2.6}), ('Ath Bilbao', 'Getafe', {'weight': 1.8}), ('Ath Bilbao', 'Hercules', {'weight': 1.5}), ('Ath Bilbao', 'La Coruna', {'weight': 1.6}), ('Ath Bilbao', 'Levante', {'weight': 1.3333333333333333}), ('Ath Bilbao', 'Malaga', {'weight': 0.8}), ('Ath Bilbao', 'Mallorca', {'weight': 1.6}), ('Ath Bilbao', 'Numancia', {'weight': 1.0}), ('Ath Bilbao', 'Osasuna', {'weight': 2.2}), ('Ath Bilbao', 'Real Madrid', {'weight': 3.0}), ('Ath Bilbao', 'Recreativo', {'weight': 1.0}), ('Ath Bilbao', 'Santander', {'weight': 1.4}), ('Ath Bilbao', 'Sevilla', {'weight': 2.4}), ('Ath Bilbao', 'Sociedad', {'weight': 0.2}), ('Ath Bilbao', 'Sp Gijon', {'weight': 0.6}), ('Ath Bilbao', 'Tenerife', {'weight': 1.4}), ('Ath Bilbao', 'Valencia', {'weight': 1.2}), ('Ath Bilbao', 'Valladolid

In [82]:
import networkx as nx
import numpy as np

# Assuming you have a matrix called 'adj_matrix' representing the adjacency matrix of the graph
# You can convert the matrix to a NetworkX graph using the 'from_numpy_array' function
# graph = nx.from_numpy_array(matrix_page_rank.T.fillna(-1).values)

# Calculate the PageRank scores using the 'pagerank' function
pagerank_scores = nx.pagerank(graph,max_iter=100, tol=1e-5)

# Access the PageRank score for each node
for node, score in pagerank_scores.items():
    print(f"Node {node}: PageRank score = {score}")

Node Ath Bilbao: PageRank score = 0.04037090540758752
Node Almeria: PageRank score = 0.031931199146513586
Node Ath Madrid: PageRank score = 0.03889193045618663
Node Barcelona: PageRank score = 0.057600983655435624
Node Betis: PageRank score = 0.03401371436092574
Node Espanol: PageRank score = 0.042706364550805495
Node Getafe: PageRank score = 0.03807781483647695
Node La Coruna: PageRank score = 0.0476556894423996
Node Levante: PageRank score = 0.032408135989923675
Node Malaga: PageRank score = 0.03638183108784246
Node Mallorca: PageRank score = 0.03921600116396907
Node Osasuna: PageRank score = 0.037324268258363456
Node Real Madrid: PageRank score = 0.05716140490043884
Node Recreativo: PageRank score = 0.02360484787123895
Node Santander: PageRank score = 0.044080988484003734
Node Sevilla: PageRank score = 0.052816924868864586
Node Sociedad: PageRank score = 0.04028760869470284
Node Sp Gijon: PageRank score = 0.03476501343392258
Node Valencia: PageRank score = 0.05077282429520663
Node V

In [105]:
pagerank_scores

{'Ath Bilbao': 0.04037090540758752,
 'Almeria': 0.031931199146513586,
 'Ath Madrid': 0.03889193045618663,
 'Barcelona': 0.057600983655435624,
 'Betis': 0.03401371436092574,
 'Espanol': 0.042706364550805495,
 'Getafe': 0.03807781483647695,
 'La Coruna': 0.0476556894423996,
 'Levante': 0.032408135989923675,
 'Malaga': 0.03638183108784246,
 'Mallorca': 0.03921600116396907,
 'Osasuna': 0.037324268258363456,
 'Real Madrid': 0.05716140490043884,
 'Recreativo': 0.02360484787123895,
 'Santander': 0.044080988484003734,
 'Sevilla': 0.052816924868864586,
 'Sociedad': 0.04028760869470284,
 'Sp Gijon': 0.03476501343392258,
 'Valencia': 0.05077282429520663,
 'Valladolid': 0.042152388027313435,
 'Villarreal': 0.04753711587863731,
 'Zaragoza': 0.028910237381441384,
 'Hercules': 0.023215814037299044,
 'Numancia': 0.031722340408664965,
 'Tenerife': 0.037455970172823434,
 'Vallecano': 0.008937683189012438}

In [83]:
pd.DataFrame(pagerank_scores.values(), pagerank_scores.keys(), columns=['page_rank']).sort_values("page_rank")

,page_rank
Vallecano,0.008938
Hercules,0.023216
Recreativo,0.023605
Zaragoza,0.028910
Numancia,0.031722
Almeria,0.031931
Levante,0.032408
Betis,0.034014
Sp Gijon,0.034765
Malaga,0.036382


## PageRank function

In [19]:
def create_graph(data):
    graph = nx.DiGraph()

    # Iterate over the DataFrame rows
    for _, row in data.iterrows():
        source = row['HomeTeam']
        target = row['AwayTeam']
        weight = row['value_5_2']
        
        # Add edges only for non-NaN values
        if not pd.isna(weight):
            graph.add_edge(source, target, weight=weight)

        source = row['AwayTeam']
        target = row['HomeTeam']
        weight = row['value_5_1']
        
        # Add edges only for non-NaN values
        if not pd.isna(weight):
            graph.add_edge(source, target, weight=weight)

    return graph

In [20]:
from sklearn.preprocessing import maxabs_scale as sc

initial_date = "1996-07-01"

team_id_name = data[["id_H","HomeTeam"]].drop_duplicates().set_index("id_H")["HomeTeam"].to_dict()


def pagerank(df,date,lag=5,div=None) -> dict:
    if div is not None: df = df[data.Div==div]
    ut.getPoints(df,"FTHG","FTAG","Points_H")
    ut.getPoints(df,"FTAG","FTHG","Points_A")

    match_sorted_list, match_sorted = get_derby_id(df,["id_H","id_A"])
    df = add_column_from_df(df,match_sorted_list,match_sorted,["id1","id2"])

    points_H = df.melt(id_vars=["derby","id_H","Date"],value_vars=["Points_H"])
    points_A = df.melt(id_vars=["derby","id_A","Date"],value_vars=["Points_A"])
    points_H.columns = points_A.columns = ["derby","id","Date","variable","value"]
    points_df = pd.concat([points_H,points_A]) 

    page_rank_matrix = ut.compute_lag(points_df,lag=lag,cols_group=["derby","id"],col_window="Date",min_samples=1,aggregations={"value":"mean"})
    page_rank_matrix = page_rank_matrix.reset_index().sort_values("Date").dropna()

    df_filtered = get_between(page_rank_matrix,initial_date,date)
    idxs = df_filtered[["derby","id"]].drop_duplicates(keep='last').index
    if len(idxs)==0: return

    page_rank_matrix_filt = page_rank_matrix.loc[idxs].drop(columns='Date')
    # page_rank_matrix_filt = page_rank_matrix_filt.set_index("derby").sort_index()
    page_rank_matrix_filt = page_rank_matrix_filt.sort_values("derby")

    page_rank_matrix_filt_1 = page_rank_matrix_filt[::2]
    page_rank_matrix_filt_2 = page_rank_matrix_filt[1::2]
    page_rank_matrix_pivoted = page_rank_matrix_filt_1.merge(page_rank_matrix_filt_2, on='derby', suffixes=('_1','_2'))

    page_rank_matrix_pivoted.loc[:,"HomeTeam"] = page_rank_matrix_pivoted.id_1.map(team_id_name)
    page_rank_matrix_pivoted.loc[:,"AwayTeam"] = page_rank_matrix_pivoted.id_2.map(team_id_name)

    graph = create_graph(page_rank_matrix_pivoted)

    pagerank_scores = nx.pagerank(graph,max_iter=100, tol=1e-5)
    pagerank_scores = pd.DataFrame(pagerank_scores.values(),pagerank_scores.keys(),columns=["pagerank"]).sort_values("pagerank")
    pagerank_scores.loc[:,"pagerank"] = sc(pagerank_scores['pagerank'],axis=0)

    return pagerank_scores

Basicamente lo que se hace es se calcula el pagerank de cada partido con un lag de N partidos (por defecto 5 partidos). Posteriormente se crea una lista de fechas comun a todos los equipos, y se busca la fecha de partido mas proxima a cada fecha comun.

In [27]:
from datetime import datetime, timedelta

def create_date_list(date1, date2, delta=2):
    """
    delta son los meses de padding entre un registro de pagerank y el siguiente
    """
    # Convert the input strings to datetime objects
    date1 = datetime.strptime(date1, "%Y-%m-%d")
    date2 = datetime.strptime(date2, "%Y-%m-%d")
    # Initialize the list of dates
    date_list = []
    # Start with the initial date
    current_date = date1
    # Add the initial date to the list
    date_list.append(current_date.strftime("%Y-%m-%d"))
    # Increment the date by 2 months until reaching the end date
    while current_date < date2:
        # Add a time delta of 2 months to the current date
        current_date += timedelta(days=delta*30)
        
        # Add the updated date to the list
        date_list.append(current_date.strftime("%Y-%m-%d"))
    return date_list

In [42]:
dict_dates_pagerank = {}

# Define the start and end dates as strings in the format "YYYY-MM-DD"
date1 = "1990-07-01"
date2 = "2023-07-01"

# Create the list of dates with a time delta of 2 months
dates = create_date_list(date1, date2,2)
lag = 5

for d in dates:
    p_rank = pagerank(old_data,d,lag=lag)
    dict_dates_pagerank[d] = p_rank
    print(d,end='\r')

In [43]:
def get_prior_date(date_list, D):
    # Filter the dates in the list that are prior to D
    prior_dates = [date for date in date_list if datetime.strptime(date, "%Y-%m-%d") < D]

    # Find the date in the prior_dates list that is closest to D
    closest_date = min(prior_dates, key=lambda date: (D - datetime.strptime(date, "%Y-%m-%d")).days)

    return closest_date

In [44]:
# get prior dates
prior_dates = get_prior_date(dates,datetime(1998,6,19))
prior_dates

'1998-05-20'

Añadimos la fecha mas proxima al partido de las creadas (comunes a todos los partidos) para posteriormente hacer el cruce de Pagerank con los datos origen. 

In [45]:
data_filt = data[data.Date>date1]
data_filt.loc[:,"Date_pagerank"] = data_filt.Date.apply(lambda x: datetime.strptime(get_prior_date(dates,x),"%Y-%m-%d"))

In [46]:
page_rank_permonth = pd.DataFrame(columns=["Team","pagerank","Date_pagerank"])
for k in dict_dates_pagerank.keys():
    if type(dict_dates_pagerank[k])==pd.DataFrame:
        dict_dates_pagerank[k].loc[:,"Date_pagerank"] = datetime.strptime(k,"%Y-%m-%d")
        dict_dates_pagerank[k].loc[:,"Team"] = dict_dates_pagerank[k].index
        dict_dates_pagerank[k] = dict_dates_pagerank[k].reset_index(drop=True)
        page_rank_permonth = pd.concat([page_rank_permonth,dict_dates_pagerank[k]])

In [47]:
page_rank_permonth

,Team,pagerank,Date_pagerank
0,Real Madrid B,0.125114,1996-09-27
1,Hercules,0.141306,1996-09-27
2,Albacete,0.176736,1996-09-27
3,Vallecano,0.208612,1996-09-27
4,Barcelona B,0.261748,1996-09-27
...,...,...,...
260,Man United,0.825773,2023-07-08
261,Paris SG,0.843549,2023-07-08
262,Inter,0.845756,2023-07-08
263,Bayern Munich,0.950498,2023-07-08


In [64]:
page_rank_permonth[page_rank_permonth.Date_pagerank=="2009-05-22"].sort_values("pagerank",ascending=False)

,Team,pagerank,Date_pagerank
210,Man United,1.000000,2009-05-22
209,Juventus,0.975215,2009-05-22
208,Inter,0.912430,2009-05-22
207,Levante,0.904466,2009-05-22
206,Arsenal,0.890497,2009-05-22
...,...,...,...
4,Burgos,0.212017,2009-05-22
3,Treviso,0.202647,2009-05-22
2,Alicante,0.163202,2009-05-22
1,Ancona,0.123417,2009-05-22


In [52]:
page_rank_permonth[page_rank_permonth.Team=="Inter"][70:130]

,Team,pagerank,Date_pagerank
203,Inter,1.000000,2008-03-28
201,Inter,0.955542,2008-05-27
201,Inter,0.955542,2008-07-26
202,Inter,0.960849,2008-09-24
201,Inter,0.980330,2008-11-23
202,Inter,0.919868,2009-01-22
208,Inter,0.958553,2009-03-23
208,Inter,0.912430,2009-05-22
207,Inter,0.890392,2009-07-21
207,Inter,0.913789,2009-09-19


In [41]:
page_rank_permonth[page_rank_permonth.Team=="Inter"][30:80]

,Team,pagerank,Date_pagerank
190,Inter,0.938427,2006-08-06
187,Inter,0.918734,2006-12-04
198,Inter,0.988300,2007-04-03
200,Inter,1.000000,2007-08-01
200,Inter,0.939599,2007-11-29
203,Inter,1.000000,2008-03-28
201,Inter,0.955542,2008-07-26
201,Inter,0.980330,2008-11-23
208,Inter,0.958553,2009-03-23
207,Inter,0.890392,2009-07-21


In [369]:
data_filt_pr = data_filt.merge(page_rank_permonth,left_on=["Date_pagerank","HomeTeam"],right_on=["Date_pagerank","Team"])
data_filt_pr = data_filt_pr.merge(page_rank_permonth,left_on=["Date_pagerank","AwayTeam"],right_on=["Date_pagerank","Team"],suffixes=("_H","_A"))
data_filt_pr = data_filt_pr[["matchId","pagerank_H","pagerank_A"]]
data_filt_pr

,matchId,pagerank_H,pagerank_A
0,0,0.750032,0.576891
1,55,0.989830,0.576891
2,42,0.737751,0.576891
3,109,0.467613,0.576891
4,135,0.721474,0.576891
...,...,...,...
49049,58092,0.436662,0.541080
49050,58166,0.682582,0.134527
49051,58085,0.535204,0.134527
49052,58017,0.318754,0.446656


In [370]:
[e for e in data_filt.Date_pagerank.unique() if e not in page_rank_permonth.Date_pagerank.unique()]

[Timestamp('1993-06-15 00:00:00'),
 Timestamp('1992-10-18 00:00:00'),
 Timestamp('1993-02-15 00:00:00'),
 Timestamp('1993-10-13 00:00:00'),
 Timestamp('1994-10-08 00:00:00'),
 Timestamp('1994-02-10 00:00:00'),
 Timestamp('1994-06-10 00:00:00'),
 Timestamp('1995-02-05 00:00:00'),
 Timestamp('1995-06-05 00:00:00'),
 Timestamp('1995-10-03 00:00:00'),
 Timestamp('1996-05-30 00:00:00'),
 Timestamp('1996-09-27 00:00:00'),
 Timestamp('1996-01-31 00:00:00'),
 Timestamp('1997-01-25 00:00:00'),
 Timestamp('1997-05-25 00:00:00')]

In [376]:
data_filt_pr.to_csv('f:\\TFG\\datasets\\raw_datasets\\page_rank.csv',sep=';',decimal=',',index=False)

# MATCH IMPORTANCE

In [236]:
# ALGORITHM:
#   1) Calcular acumulado de puntos por equipo-temporada
#   2) Calcular jornada de cada partido
#   2) Tomar los 5 primeros y 5 ultimos -> guardarlo en una lista
        # a) Una vez calculados los puntos acumulados, y por cada partido:
            #   calcular el rank de puntos agrupando por Div-jornada.
#   3) Aplicar la formula de (rank(k) - rank(i)) / np -> np (partidos jugados), rank(k)
        # a) Columna de np (por cada equipo-temporada)
        # b) Columna de rank(i) / np
        # c) Columna de rank(k) / np
        # d) Columna aplicando la formula restando b) y c)
        # e) Repetir para k={1,2,3,4,5,-5,-4,-3,-2,-1}

In [237]:
COLS_H      = ['matchId','Div','Date','season','id_H','HomeTeam','FTHG','FTAG']
COLS_A      = ['matchId','Div','Date','season','id_A','AwayTeam','FTAG','FTHG']
COLS_AUX    = ['matchId','Div','Date','season','idTeam','Team','FTG','FTG_rival']

data_split = ut.split_data_side(data,COLS_H,COLS_A,COLS_AUX)
data_split.loc[:,"rounds"] = 1
data_split

f:\TFG\code\experiments\utils.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_H.loc[:,'Side'] = 0
f:\TFG\code\experiments\utils.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,new_col] = 1
f:\TFG\code\experiments\utils.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returni

,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,rounds
6316,6316,D1,1993-01-09,T93-94,2,Bayern Munich,3.0,0.0,0,3,1
6316,6316,D1,1993-01-09,T93-94,25,Leipzig,0.0,3.0,1,0,1
6317,6317,D1,1993-01-09,T93-94,8,Dortmund,4.0,0.0,0,3,1
6317,6317,D1,1993-01-09,T93-94,9,Dresden,0.0,4.0,1,0,1
6318,6318,D1,1993-01-09,T93-94,12,Ein Frankfurt,3.0,1.0,0,3,1
...,...,...,...,...,...,...,...,...,...,...,...
44863,44863,SP1,2021-12-05,T20-21,200,Getafe,0,1,1,0,1
44864,44864,SP1,2021-12-05,T20-21,205,Huesca,1,0,0,3,1
44864,44864,SP1,2021-12-05,T20-21,188,Ath Bilbao,0,1,1,0,1
44865,44865,SP1,2021-12-05,T20-21,189,Ath Madrid,2,1,0,3,1


In [238]:
rounds = data_split.sort_values("Date").groupby(["Div","idTeam","season"]).expanding().agg({"rounds":"sum","Points":"mean"}).reset_index()
rounds.columns = ["Div","idTeam","season","matchId","rounds","Points_mean"]
rounds

,Div,idTeam,season,matchId,rounds,Points_mean
0,D1,0,T07-08,2147,1.0,0.000000
1,D1,0,T07-08,2096,2.0,0.000000
2,D1,0,T07-08,2104,3.0,0.333333
3,D1,0,T07-08,2024,4.0,0.250000
4,D1,0,T07-08,2035,5.0,0.800000
...,...,...,...,...,...,...
117087,SP2,317,T20-21,56582,34.0,1.088235
117088,SP2,317,T20-21,56771,35.0,1.085714
117089,SP2,317,T20-21,56728,36.0,1.138889
117090,SP2,317,T20-21,56625,37.0,1.135135


In [239]:
rounds.loc[:,"rank"] = rounds.groupby(["Div","season","rounds"]).Points_mean.rank(method='first',ascending=False)
rounds

,Div,idTeam,season,matchId,rounds,Points_mean,rank
0,D1,0,T07-08,2147,1.0,0.000000,14.0
1,D1,0,T07-08,2096,2.0,0.000000,17.0
2,D1,0,T07-08,2104,3.0,0.333333,14.0
3,D1,0,T07-08,2024,4.0,0.250000,16.0
4,D1,0,T07-08,2035,5.0,0.800000,14.0
...,...,...,...,...,...,...,...
117087,SP2,317,T20-21,56582,34.0,1.088235,19.0
117088,SP2,317,T20-21,56771,35.0,1.085714,18.0
117089,SP2,317,T20-21,56728,36.0,1.138889,16.0
117090,SP2,317,T20-21,56625,37.0,1.135135,15.0


In [240]:
rounds[(rounds.Div=="SP1") & (rounds.season=='T19-20') & (rounds.rounds==14)].sort_values("rank")

,Div,idTeam,season,matchId,rounds,Points_mean,rank
77641,SP1,190,T19-20,44229,14.0,2.214286,1.0
87714,SP1,219,T19-20,44262,14.0,2.214286,2.0
89687,SP1,223,T19-20,44226,14.0,1.928571,3.0
76580,SP1,189,T19-20,44222,14.0,1.714286,4.0
90556,SP1,224,T19-20,44230,14.0,1.714286,5.0
81946,SP1,200,T19-20,44226,14.0,1.642857,6.0
82364,SP1,203,T19-20,44224,14.0,1.642857,7.0
92354,SP1,227,T19-20,44231,14.0,1.642857,8.0
86576,SP1,217,T19-20,44227,14.0,1.571429,9.0
75595,SP1,188,T19-20,44222,14.0,1.428571,10.0


In [241]:
data_split = data_split.drop(columns="rounds").merge(rounds, on=["Div","idTeam","season","matchId"])
data_split

,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,rounds,Points_mean,rank
0,6316,D1,1993-01-09,T93-94,2,Bayern Munich,3.0,0.0,0,3,1.0,3.000000,1.0
1,6316,D1,1993-01-09,T93-94,25,Leipzig,0.0,3.0,1,0,1.0,0.000000,13.0
2,6317,D1,1993-01-09,T93-94,8,Dortmund,4.0,0.0,0,3,1.0,3.000000,2.0
3,6317,D1,1993-01-09,T93-94,9,Dresden,0.0,4.0,1,0,1.0,0.000000,10.0
4,6318,D1,1993-01-09,T93-94,12,Ein Frankfurt,3.0,1.0,0,3,1.0,3.000000,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
117087,44863,SP1,2021-12-05,T20-21,200,Getafe,0,1,1,0,38.0,1.000000,16.0
117088,44864,SP1,2021-12-05,T20-21,205,Huesca,1,0,0,3,38.0,0.894737,18.0
117089,44864,SP1,2021-12-05,T20-21,188,Ath Bilbao,0,1,1,0,38.0,1.210526,9.0
117090,44865,SP1,2021-12-05,T20-21,189,Ath Madrid,2,1,0,3,38.0,2.263158,1.0


In [242]:
rounds = rounds[["Div","season","rounds","rank","Points_mean"]]
rounds

,Div,season,rounds,rank,Points_mean
0,D1,T07-08,1.0,14.0,0.000000
1,D1,T07-08,2.0,17.0,0.000000
2,D1,T07-08,3.0,14.0,0.333333
3,D1,T07-08,4.0,16.0,0.250000
4,D1,T07-08,5.0,14.0,0.800000
...,...,...,...,...,...
117087,SP2,T20-21,34.0,19.0,1.088235
117088,SP2,T20-21,35.0,18.0,1.085714
117089,SP2,T20-21,36.0,16.0,1.138889
117090,SP2,T20-21,37.0,15.0,1.135135


In [243]:
data_rounds = data_split.merge(rounds,on=["Div","season","rounds"],suffixes=("","_others"))
data_rounds

,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,rounds,Points_mean,rank,rank_others,Points_mean_others
0,6316,D1,1993-01-09,T93-94,2,Bayern Munich,3.0,0.0,0,3,1.0,3.000000,1.0,1.0,3.000000
1,6316,D1,1993-01-09,T93-94,2,Bayern Munich,3.0,0.0,0,3,1.0,3.000000,1.0,2.0,3.000000
2,6316,D1,1993-01-09,T93-94,2,Bayern Munich,3.0,0.0,0,3,1.0,3.000000,1.0,10.0,0.000000
3,6316,D1,1993-01-09,T93-94,2,Bayern Munich,3.0,0.0,0,3,1.0,3.000000,1.0,3.0,3.000000
4,6316,D1,1993-01-09,T93-94,2,Bayern Munich,3.0,0.0,0,3,1.0,3.000000,1.0,4.0,3.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2335337,34878,I1,2021-12-05,T20-21,160,Milan,7,0,1,3,36.0,2.083333,3.0,6.0,1.638889
2335338,34878,I1,2021-12-05,T20-21,160,Milan,7,0,1,3,36.0,2.083333,3.0,8.0,1.277778
2335339,34878,I1,2021-12-05,T20-21,160,Milan,7,0,1,3,36.0,2.083333,3.0,7.0,1.555556
2335340,34878,I1,2021-12-05,T20-21,160,Milan,7,0,1,3,36.0,2.083333,3.0,14.0,0.972222


In [244]:
data_rounds.columns

Index(['matchId', 'Div', 'Date', 'season', 'idTeam', 'Team', 'FTG',
       'FTG_rival', 'Side', 'Points', 'rounds', 'Points_mean', 'rank',
       'rank_others', 'Points_mean_others'],
      dtype='object')

In [245]:
data_rounds = data_rounds.pivot(index=['matchId', 'Div', 'Date', 'season','idTeam', 'Team', 'FTG',
       'FTG_rival', 'Side', 'Points', 'rounds', 'Points_mean', 'rank'],columns='rank_others',values='Points_mean_others')
data_rounds = data_rounds.reset_index()
data_rounds

rank_others,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,...,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0
0,0,D1,2000-11-08,T00-01,8,Dortmund,1,0,0,3,...,1.142857,1.071429,1.000000,1.000000,1.000000,0.857143,NaN,NaN,NaN,NaN
1,0,D1,2000-11-08,T00-01,19,Hansa Rostock,0,1,1,0,...,1.076923,1.076923,1.076923,1.000000,0.846154,0.846154,NaN,NaN,NaN,NaN
2,1,D1,2000-12-08,T00-01,2,Bayern Munich,4,1,0,3,...,1.125000,1.125000,1.062500,1.062500,0.937500,0.875000,NaN,NaN,NaN,NaN
3,1,D1,2000-12-08,T00-01,20,Hertha,1,4,1,0,...,1.125000,1.125000,1.062500,1.062500,0.937500,0.875000,NaN,NaN,NaN,NaN
4,2,D1,2000-12-08,T00-01,15,Freiburg,4,0,0,3,...,1.125000,1.125000,1.062500,1.062500,0.937500,0.875000,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117087,58543,SP2,2000-04-06,T99-00,314,Villarreal,0.0,1.0,1,0,...,1.300000,1.300000,1.266667,1.266667,1.233333,1.133333,1.100000,1.000000,1.000000,0.866667
117088,58544,SP2,2000-04-06,T99-00,288,Osasuna,2.0,1.0,0,3,...,1.300000,1.300000,1.266667,1.266667,1.233333,1.133333,1.100000,1.000000,1.000000,0.866667
117089,58544,SP2,2000-04-06,T99-00,297,Recreativo,1.0,2.0,1,0,...,1.300000,1.300000,1.266667,1.266667,1.233333,1.133333,1.100000,1.000000,1.000000,0.866667
117090,58545,SP2,2000-04-06,T99-00,255,Eibar,1.0,2.0,1,0,...,1.290323,1.290323,1.258065,1.258065,1.258065,1.161290,1.129032,1.064516,0.967742,0.870968


In [246]:
for c in range(1,21):
    data_rounds.loc[:,c] = data_rounds.loc[:,c] - data_rounds.loc[:,"Points_mean"] 

data_rounds

rank_others,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,...,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0
0,0,D1,2000-11-08,T00-01,8,Dortmund,1,0,0,3,...,-0.642857,-0.714286,-0.785714,-0.785714,-0.785714,-0.928571,NaN,NaN,NaN,NaN
1,0,D1,2000-11-08,T00-01,19,Hansa Rostock,0,1,1,0,...,0.000000,0.000000,0.000000,-0.076923,-0.230769,-0.230769,NaN,NaN,NaN,NaN
2,1,D1,2000-12-08,T00-01,2,Bayern Munich,4,1,0,3,...,-0.562500,-0.562500,-0.625000,-0.625000,-0.750000,-0.812500,NaN,NaN,NaN,NaN
3,1,D1,2000-12-08,T00-01,20,Hertha,1,4,1,0,...,-0.625000,-0.625000,-0.687500,-0.687500,-0.812500,-0.875000,NaN,NaN,NaN,NaN
4,2,D1,2000-12-08,T00-01,15,Freiburg,4,0,0,3,...,-0.125000,-0.125000,-0.187500,-0.187500,-0.312500,-0.375000,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117087,58543,SP2,2000-04-06,T99-00,314,Villarreal,0.0,1.0,1,0,...,-0.166667,-0.166667,-0.200000,-0.200000,-0.233333,-0.333333,-0.366667,-0.466667,1.000000,0.866667
117088,58544,SP2,2000-04-06,T99-00,288,Osasuna,2.0,1.0,0,3,...,-0.133333,-0.133333,-0.166667,-0.166667,-0.200000,-0.300000,-0.333333,-0.433333,1.000000,0.866667
117089,58544,SP2,2000-04-06,T99-00,297,Recreativo,1.0,2.0,1,0,...,0.300000,0.300000,0.266667,0.266667,0.233333,0.133333,0.100000,0.000000,1.000000,0.866667
117090,58545,SP2,2000-04-06,T99-00,255,Eibar,1.0,2.0,1,0,...,-0.032258,-0.032258,-0.064516,-0.064516,-0.064516,-0.161290,-0.193548,-0.258065,0.967742,0.870968


In [247]:
match_importance = data_rounds[['matchId','Div','Date','season','idTeam','Team','FTG','FTG_rival','Side','Points','rounds','rank']].copy()
match_importance

rank_others,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,rounds,rank
0,0,D1,2000-11-08,T00-01,8,Dortmund,1,0,0,3,14.0,1.0
1,0,D1,2000-11-08,T00-01,19,Hansa Rostock,0,1,1,0,13.0,13.0
2,1,D1,2000-12-08,T00-01,2,Bayern Munich,4,1,0,3,16.0,5.0
3,1,D1,2000-12-08,T00-01,20,Hertha,1,4,1,0,16.0,4.0
4,2,D1,2000-12-08,T00-01,15,Freiburg,4,0,0,3,16.0,11.0
...,...,...,...,...,...,...,...,...,...,...,...,...
117087,58543,SP2,2000-04-06,T99-00,314,Villarreal,0.0,1.0,1,0,30.0,9.0
117088,58544,SP2,2000-04-06,T99-00,288,Osasuna,2.0,1.0,0,3,30.0,10.0
117089,58544,SP2,2000-04-06,T99-00,297,Recreativo,1.0,2.0,1,0,30.0,21.0
117090,58545,SP2,2000-04-06,T99-00,255,Eibar,1.0,2.0,1,0,31.0,12.0


In [248]:
# take top 5
match_importance.loc[:,"top1"] = data_rounds.loc[:,1]
match_importance.loc[:,"top2"] = data_rounds.loc[:,2]
match_importance.loc[:,"top3"] = data_rounds.loc[:,3]
match_importance.loc[:,"top4"] = data_rounds.loc[:,4]
match_importance.loc[:,"top5"] = data_rounds.loc[:,5]

match_importance

rank_others,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,rounds,rank,top1,top2,top3,top4,top5
0,0,D1,2000-11-08,T00-01,8,Dortmund,1,0,0,3,14.0,1.0,0.000000,0.000000,0.000000,-0.071429,-0.071429
1,0,D1,2000-11-08,T00-01,19,Hansa Rostock,0,1,1,0,13.0,13.0,0.846154,0.769231,0.692308,0.692308,0.615385
2,1,D1,2000-12-08,T00-01,2,Bayern Munich,4,1,0,3,16.0,5.0,0.250000,0.187500,0.125000,0.062500,0.000000
3,1,D1,2000-12-08,T00-01,20,Hertha,1,4,1,0,16.0,4.0,0.187500,0.125000,0.062500,0.000000,-0.062500
4,2,D1,2000-12-08,T00-01,15,Freiburg,4,0,0,3,16.0,11.0,0.687500,0.625000,0.562500,0.500000,0.437500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117087,58543,SP2,2000-04-06,T99-00,314,Villarreal,0.0,1.0,1,0,30.0,9.0,0.200000,0.133333,0.100000,0.100000,0.100000
117088,58544,SP2,2000-04-06,T99-00,288,Osasuna,2.0,1.0,0,3,30.0,10.0,0.233333,0.166667,0.133333,0.133333,0.133333
117089,58544,SP2,2000-04-06,T99-00,297,Recreativo,1.0,2.0,1,0,30.0,21.0,0.666667,0.600000,0.566667,0.566667,0.566667
117090,58545,SP2,2000-04-06,T99-00,255,Eibar,1.0,2.0,1,0,31.0,12.0,0.322581,0.290323,0.258065,0.258065,0.225806


In [249]:
# take bottom 5

# rows w/ 22 teams

idx22 = data_rounds[22].isna().values
idx20 = data_rounds[20].isna().values
idx18 = data_rounds[18].isna().values
idx16 = data_rounds[16].isna().values

for pos,idx in zip([16,18,20,22],[idx16,idx18,idx20,idx22]):
    match_importance.loc[~idx,"down1"] = data_rounds.loc[~idx,pos]
    match_importance.loc[~idx,"down2"] = data_rounds.loc[~idx,pos-1]
    match_importance.loc[~idx,"down3"] = data_rounds.loc[~idx,pos-2]
    match_importance.loc[~idx,"down4"] = data_rounds.loc[~idx,pos-3]
    match_importance.loc[~idx,"down5"] = data_rounds.loc[~idx,pos-4]
    match_importance.loc[~idx,"_edited"] = pos

In [250]:
match_importance

rank_others,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,...,top2,top3,top4,top5,down1,down2,down3,down4,down5,_edited
0,0,D1,2000-11-08,T00-01,8,Dortmund,1,0,0,3,...,0.000000,0.000000,-0.071429,-0.071429,-0.928571,-0.785714,-0.785714,-0.785714,-0.714286,18.0
1,0,D1,2000-11-08,T00-01,19,Hansa Rostock,0,1,1,0,...,0.769231,0.692308,0.692308,0.615385,-0.230769,-0.230769,-0.076923,0.000000,0.000000,18.0
2,1,D1,2000-12-08,T00-01,2,Bayern Munich,4,1,0,3,...,0.187500,0.125000,0.062500,0.000000,-0.812500,-0.750000,-0.625000,-0.625000,-0.562500,18.0
3,1,D1,2000-12-08,T00-01,20,Hertha,1,4,1,0,...,0.125000,0.062500,0.000000,-0.062500,-0.875000,-0.812500,-0.687500,-0.687500,-0.625000,18.0
4,2,D1,2000-12-08,T00-01,15,Freiburg,4,0,0,3,...,0.625000,0.562500,0.500000,0.437500,-0.375000,-0.312500,-0.187500,-0.187500,-0.125000,18.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117087,58543,SP2,2000-04-06,T99-00,314,Villarreal,0.0,1.0,1,0,...,0.133333,0.100000,0.100000,0.100000,0.866667,1.000000,-0.466667,-0.366667,-0.333333,22.0
117088,58544,SP2,2000-04-06,T99-00,288,Osasuna,2.0,1.0,0,3,...,0.166667,0.133333,0.133333,0.133333,0.866667,1.000000,-0.433333,-0.333333,-0.300000,22.0
117089,58544,SP2,2000-04-06,T99-00,297,Recreativo,1.0,2.0,1,0,...,0.600000,0.566667,0.566667,0.566667,0.866667,1.000000,0.000000,0.100000,0.133333,22.0
117090,58545,SP2,2000-04-06,T99-00,255,Eibar,1.0,2.0,1,0,...,0.290323,0.258065,0.258065,0.225806,0.870968,0.967742,-0.258065,-0.193548,-0.161290,22.0


In [62]:
def merge_sides(df,group_on,col_order):
    mask_home = df.Side==0
    mask_away = df.Side==1
    data_merged = df[mask_home].merge(df[mask_away], on=group_on, suffixes=("_H","_A"))
    data_merged = data_merged[col_order]
    return data_merged

In [252]:
match_importance.columns

Index(['matchId', 'Div', 'Date', 'season', 'idTeam', 'Team', 'FTG',
       'FTG_rival', 'Side', 'Points', 'rounds', 'rank', 'top1', 'top2', 'top3',
       'top4', 'top5', 'down1', 'down2', 'down3', 'down4', 'down5', '_edited'],
      dtype='object', name='rank_others')

In [253]:
group_on = ['matchId', 'Div', 'Date', 'season']
feats = [ [f+s for f in ['top1', 'top2', 'top3','top4', 'top5', 'down1', 'down2', 'down3', 'down4', 'down5']] for s in ['_H','_A']  ]
feats = [*feats[0],*feats[1]]
col_order = [*group_on, 'idTeam_H','idTeam_A','Team_H','Team_A',"rounds_H","rounds_A","rank_H","rank_A",*feats,"_edited_H","_edited_A"]
match_importance = merge_sides(match_importance.drop(columns=["FTG","FTG_rival"]),group_on,col_order)
match_importance

rank_others,matchId,Div,Date,season,idTeam_H,idTeam_A,Team_H,Team_A,rounds_H,rounds_A,...,top3_A,top4_A,top5_A,down1_A,down2_A,down3_A,down4_A,down5_A,_edited_H,_edited_A
0,0,D1,2000-11-08,T00-01,8,19,Dortmund,Hansa Rostock,14.0,13.0,...,0.692308,0.692308,0.615385,-0.230769,-0.230769,-0.076923,0.000000,0.000000,18.0,18.0
1,1,D1,2000-12-08,T00-01,2,20,Bayern Munich,Hertha,16.0,16.0,...,0.062500,0.000000,-0.062500,-0.875000,-0.812500,-0.687500,-0.687500,-0.625000,18.0,18.0
2,2,D1,2000-12-08,T00-01,15,35,Freiburg,Stuttgart,16.0,15.0,...,0.866667,0.800000,0.666667,-0.133333,0.000000,0.000000,0.066667,0.200000,18.0,18.0
3,3,D1,2000-12-08,T00-01,17,29,Hamburg,Munich 1860,16.0,16.0,...,0.687500,0.625000,0.562500,-0.250000,-0.187500,-0.062500,-0.062500,0.000000,18.0,18.0
4,4,D1,2000-12-08,T00-01,23,4,Kaiserslautern,Bochum,15.0,16.0,...,0.875000,0.812500,0.750000,-0.062500,0.000000,0.125000,0.125000,0.187500,18.0,18.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,58541,SP2,2000-04-06,T99-00,258,243,Extremadura,Badajoz,30.0,29.0,...,0.344828,0.344828,0.344828,0.896552,1.000000,-0.206897,-0.137931,-0.103448,22.0,22.0
58542,58542,SP2,2000-04-06,T99-00,271,274,Las Palmas,Levante,30.0,30.0,...,0.033333,0.033333,0.033333,0.866667,1.000000,-0.533333,-0.433333,-0.400000,22.0,22.0
58543,58543,SP2,2000-04-06,T99-00,277,314,Logrones,Villarreal,30.0,30.0,...,0.100000,0.100000,0.100000,0.866667,1.000000,-0.466667,-0.366667,-0.333333,22.0,22.0
58544,58544,SP2,2000-04-06,T99-00,288,297,Osasuna,Recreativo,30.0,30.0,...,0.566667,0.566667,0.566667,0.866667,1.000000,0.000000,0.100000,0.133333,22.0,22.0


In [254]:
col_order = [*group_on, 'idTeam_H','idTeam_A','Team_H','Team_A',"rounds","rank_H","rank_A",*feats,"_edited_H","_edited_A"]

match_importance.loc[:,"rounds"] = match_importance[["rounds_H","rounds_A"]].mean(axis=1)
match_importance = match_importance.drop(columns=["rounds_H","rounds_A"])
match_importance

rank_others,matchId,Div,Date,season,idTeam_H,idTeam_A,Team_H,Team_A,rank_H,rank_A,...,top4_A,top5_A,down1_A,down2_A,down3_A,down4_A,down5_A,_edited_H,_edited_A,rounds
0,0,D1,2000-11-08,T00-01,8,19,Dortmund,Hansa Rostock,1.0,13.0,...,0.692308,0.615385,-0.230769,-0.230769,-0.076923,0.000000,0.000000,18.0,18.0,13.5
1,1,D1,2000-12-08,T00-01,2,20,Bayern Munich,Hertha,5.0,4.0,...,0.000000,-0.062500,-0.875000,-0.812500,-0.687500,-0.687500,-0.625000,18.0,18.0,16.0
2,2,D1,2000-12-08,T00-01,15,35,Freiburg,Stuttgart,11.0,17.0,...,0.800000,0.666667,-0.133333,0.000000,0.000000,0.066667,0.200000,18.0,18.0,15.5
3,3,D1,2000-12-08,T00-01,17,29,Hamburg,Munich 1860,9.0,14.0,...,0.625000,0.562500,-0.250000,-0.187500,-0.062500,-0.062500,0.000000,18.0,18.0,16.0
4,4,D1,2000-12-08,T00-01,23,4,Kaiserslautern,Bochum,6.0,17.0,...,0.812500,0.750000,-0.062500,0.000000,0.125000,0.125000,0.187500,18.0,18.0,15.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,58541,SP2,2000-04-06,T99-00,258,243,Extremadura,Badajoz,3.0,15.0,...,0.344828,0.344828,0.896552,1.000000,-0.206897,-0.137931,-0.103448,22.0,22.0,29.5
58542,58542,SP2,2000-04-06,T99-00,271,274,Las Palmas,Levante,2.0,6.0,...,0.033333,0.033333,0.866667,1.000000,-0.533333,-0.433333,-0.400000,22.0,22.0,30.0
58543,58543,SP2,2000-04-06,T99-00,277,314,Logrones,Villarreal,14.0,9.0,...,0.100000,0.100000,0.866667,1.000000,-0.466667,-0.366667,-0.333333,22.0,22.0,30.0
58544,58544,SP2,2000-04-06,T99-00,288,297,Osasuna,Recreativo,10.0,21.0,...,0.566667,0.566667,0.866667,1.000000,0.000000,0.100000,0.133333,22.0,22.0,30.0


In [257]:
match_importance[["matchId","rounds",*feats]].to_csv('f:\\TFG\\datasets\\raw_datasets\\match_importance.csv',sep=';',decimal=',',index=False)
match_importance[["matchId","rounds",*feats]].dropna()

rank_others,matchId,rounds,top1_H,top2_H,top3_H,top4_H,top5_H,down1_H,down2_H,down3_H,...,top1_A,top2_A,top3_A,top4_A,top5_A,down1_A,down2_A,down3_A,down4_A,down5_A
0,0,13.5,0.000000,0.000000,0.000000,-0.071429,-0.071429,-0.928571,-0.785714,-0.785714,...,0.846154,0.769231,0.692308,0.692308,0.615385,-0.230769,-0.230769,-0.076923,0.000000,0.000000
1,1,16.0,0.250000,0.187500,0.125000,0.062500,0.000000,-0.812500,-0.750000,-0.625000,...,0.187500,0.125000,0.062500,0.000000,-0.062500,-0.875000,-0.812500,-0.687500,-0.687500,-0.625000
2,2,15.5,0.687500,0.625000,0.562500,0.500000,0.437500,-0.375000,-0.312500,-0.187500,...,0.933333,0.933333,0.866667,0.800000,0.666667,-0.133333,0.000000,0.000000,0.066667,0.200000
3,3,16.0,0.625000,0.562500,0.500000,0.437500,0.375000,-0.437500,-0.375000,-0.250000,...,0.812500,0.750000,0.687500,0.625000,0.562500,-0.250000,-0.187500,-0.062500,-0.062500,0.000000
4,4,15.5,0.266667,0.266667,0.200000,0.133333,0.000000,-0.800000,-0.666667,-0.666667,...,1.000000,0.937500,0.875000,0.812500,0.750000,-0.062500,0.000000,0.125000,0.125000,0.187500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,58541,29.5,0.100000,0.033333,0.000000,0.000000,0.000000,0.866667,1.000000,-0.566667,...,0.379310,0.379310,0.344828,0.344828,0.344828,0.896552,1.000000,-0.206897,-0.137931,-0.103448
58542,58542,30.0,0.066667,0.000000,-0.033333,-0.033333,-0.033333,0.866667,1.000000,-0.600000,...,0.133333,0.066667,0.033333,0.033333,0.033333,0.866667,1.000000,-0.533333,-0.433333,-0.400000
58543,58543,30.0,0.366667,0.300000,0.266667,0.266667,0.266667,0.866667,1.000000,-0.300000,...,0.200000,0.133333,0.100000,0.100000,0.100000,0.866667,1.000000,-0.466667,-0.366667,-0.333333
58544,58544,30.0,0.233333,0.166667,0.133333,0.133333,0.133333,0.866667,1.000000,-0.433333,...,0.666667,0.600000,0.566667,0.566667,0.566667,0.866667,1.000000,0.000000,0.100000,0.133333


In [258]:
rounds[(rounds.Div=="D1") & (rounds.season=='T02-03') & (rounds.rounds==33)].sort_values("rank")

,Div,season,rounds,rank,Points_mean
7714,D1,T02-03,33.0,1.0,1.545455
12999,D1,T02-03,33.0,2.0,1.393939
1708,D1,T02-03,33.0,3.0,1.363636
8952,D1,T02-03,33.0,4.0,1.181818
7352,D1,T02-03,33.0,5.0,1.151515
2170,D1,T02-03,33.0,6.0,0.818182


In [1]:
['1','2'] + ['3']

['1', '2', '3']

In [259]:
df = match_importance
df[df.isna().any(axis=1)]

rank_others,matchId,Div,Date,season,idTeam_H,idTeam_A,Team_H,Team_A,rank_H,rank_A,...,top4_A,top5_A,down1_A,down2_A,down3_A,down4_A,down5_A,_edited_H,_edited_A,rounds
842,842,D1,2003-12-04,T02-03,20,4,Hertha,Bochum,1.0,3.0,...,-0.181818,-0.212121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.0
843,843,D1,2003-12-04,T02-03,23,19,Kaiserslautern,Hansa Rostock,4.0,5.0,...,0.030303,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.0
879,879,D1,2003-10-05,T02-03,29,6,Munich 1860,Cottbus,10.0,6.0,...,0.363636,0.333333,NaN,NaN,NaN,NaN,NaN,18.0,NaN,32.0
883,883,D1,2003-11-05,T02-03,33,18,Schalke 04,Hannover,2.0,8.0,...,0.281250,0.156250,-0.5,-0.5,-0.25,-0.21875,-0.1875,NaN,16.0,32.5
1001,1001,D1,2004-07-02,T03-04,19,15,Hansa Rostock,Freiburg,7.0,6.0,...,0.136364,0.045455,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50757,50757,SP2,2008-11-05,T07-08,233,234,Alaves,Albacete,11.0,11.0,...,0.219512,0.121951,NaN,NaN,NaN,NaN,NaN,20.0,NaN,40.5
50759,50759,SP2,2008-11-05,T07-08,253,316,Cordoba,Xerez,18.0,6.0,...,0.146341,0.048780,NaN,NaN,NaN,NaN,NaN,20.0,NaN,40.5
50760,50760,SP2,2008-11-05,T07-08,265,259,Granada 74,Ferrol,20.0,10.0,...,0.195122,0.097561,NaN,NaN,NaN,NaN,NaN,20.0,NaN,40.5
50761,50761,SP2,2008-11-05,T07-08,267,291,Hercules,Poli Ejido,8.0,13.0,...,0.341463,0.243902,NaN,NaN,NaN,NaN,NaN,22.0,NaN,40.0


# HISTORICAL STRENGTH

Remove the first 5 games of each season.

In [48]:
data

,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Points_H,Points_A
0,0,8,19,D1,0,2000-11-08,T00-01,Dortmund,Hansa Rostock,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
1,1,2,20,D1,1,2000-12-08,T00-01,Bayern Munich,Hertha,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
2,2,15,35,D1,2,2000-12-08,T00-01,Freiburg,Stuttgart,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
3,3,17,29,D1,3,2000-12-08,T00-01,Hamburg,Munich 1860,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
4,4,23,4,D1,4,2000-12-08,T00-01,Kaiserslautern,Bochum,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,58541,258,243,SP2,10828,2000-04-06,T99-00,Extremadura,Badajoz,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
58542,58542,271,274,SP2,10829,2000-04-06,T99-00,Las Palmas,Levante,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
58543,58543,277,314,SP2,10830,2000-04-06,T99-00,Logrones,Villarreal,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
58544,58544,288,297,SP2,10831,2000-04-06,T99-00,Osasuna,Recreativo,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0


In [49]:
old_data = data.copy()
ut.getPoints(data,"FTHG","FTAG","Points_H")
ut.getPoints(data,"FTAG","FTHG","Points_A")
data

,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Points_H,Points_A
0,0,8,19,D1,0,2000-11-08,T00-01,Dortmund,Hansa Rostock,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
1,1,2,20,D1,1,2000-12-08,T00-01,Bayern Munich,Hertha,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
2,2,15,35,D1,2,2000-12-08,T00-01,Freiburg,Stuttgart,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
3,3,17,29,D1,3,2000-12-08,T00-01,Hamburg,Munich 1860,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
4,4,23,4,D1,4,2000-12-08,T00-01,Kaiserslautern,Bochum,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,58541,258,243,SP2,10828,2000-04-06,T99-00,Extremadura,Badajoz,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
58542,58542,271,274,SP2,10829,2000-04-06,T99-00,Las Palmas,Levante,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
58543,58543,277,314,SP2,10830,2000-04-06,T99-00,Logrones,Villarreal,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
58544,58544,288,297,SP2,10831,2000-04-06,T99-00,Osasuna,Recreativo,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0


In [50]:
COLS_H      = ['matchId','Div','Date','season','id_H','HomeTeam','FTHG','FTAG']
COLS_A      = ['matchId','Div','Date','season','id_A','AwayTeam','FTAG','FTHG']
COLS_AUX    = ['matchId','Div','Date','season','idTeam','Team','FTG','FTG_rival']

data_split = ut.split_data_side(data,COLS_H,COLS_A,COLS_AUX)
data_split

f:\TFG\code\experiments\utils.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_H.loc[:,'Side'] = 0
f:\TFG\code\experiments\utils.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,new_col] = 1
f:\TFG\code\experiments\utils.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returni

,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points
6316,6316,D1,1993-01-09,T93-94,2,Bayern Munich,3.0,0.0,0,3
6316,6316,D1,1993-01-09,T93-94,25,Leipzig,0.0,3.0,1,0
6317,6317,D1,1993-01-09,T93-94,8,Dortmund,4.0,0.0,0,3
6317,6317,D1,1993-01-09,T93-94,9,Dresden,0.0,4.0,1,0
6318,6318,D1,1993-01-09,T93-94,12,Ein Frankfurt,3.0,1.0,0,3
...,...,...,...,...,...,...,...,...,...,...
44863,44863,SP1,2021-12-05,T20-21,200,Getafe,0,1,1,0
44864,44864,SP1,2021-12-05,T20-21,205,Huesca,1,0,0,3
44864,44864,SP1,2021-12-05,T20-21,188,Ath Bilbao,0,1,1,0
44865,44865,SP1,2021-12-05,T20-21,189,Ath Madrid,2,1,0,3


In [51]:
pd.get_dummies(data_split.Points)

,0,1,3
6316,False,False,True
6316,True,False,False
6317,False,False,True
6317,True,False,False
6318,False,False,True
...,...,...,...
44863,True,False,False
44864,False,False,True
44864,True,False,False
44865,False,False,True


In [52]:
data_split[["Loss","Draw","Win"]] = pd.get_dummies(data_split.Points)
data_split

,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,Loss,Draw,Win
6316,6316,D1,1993-01-09,T93-94,2,Bayern Munich,3.0,0.0,0,3,False,False,True
6316,6316,D1,1993-01-09,T93-94,25,Leipzig,0.0,3.0,1,0,True,False,False
6317,6317,D1,1993-01-09,T93-94,8,Dortmund,4.0,0.0,0,3,False,False,True
6317,6317,D1,1993-01-09,T93-94,9,Dresden,0.0,4.0,1,0,True,False,False
6318,6318,D1,1993-01-09,T93-94,12,Ein Frankfurt,3.0,1.0,0,3,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
44863,44863,SP1,2021-12-05,T20-21,200,Getafe,0,1,1,0,True,False,False
44864,44864,SP1,2021-12-05,T20-21,205,Huesca,1,0,0,3,False,False,True
44864,44864,SP1,2021-12-05,T20-21,188,Ath Bilbao,0,1,1,0,True,False,False
44865,44865,SP1,2021-12-05,T20-21,189,Ath Madrid,2,1,0,3,False,False,True


In [53]:
ut.compute_lags

<function utils.compute_lags(df, lags, min_samples, col_window, cols_group, aggregations, cols_agg=False)>

In [76]:
aggs = {"Draw":["mean"],"Win":["mean"],"FTG":["mean","std"],"FTG_rival":["mean","std"]}
cols_aggs = ["Draw","Win","FTG_mean","FTG_std","FTG_rival_mean","FTG_rival_std"]
long_term = ut.compute_lag(data_split,"730D",5,'Date',["idTeam","Team","Side"],aggs,cols_aggs)
long_term

Draw       Win  FTG_mean   FTG_std   
idTeam Team     Side Date                                                 
0      Aachen   0    2006-03-12       NaN       NaN       NaN       NaN  \
                     2006-04-11       NaN       NaN       NaN       NaN   
                     2006-08-19       NaN       NaN       NaN       NaN   
                     2006-09-16       NaN       NaN       NaN       NaN   
                     2006-09-30       NaN       NaN       NaN       NaN   
...                                   ...       ...       ...       ...   
317    Zaragoza 1    2021-04-30  0.261905  0.285714  1.023810  0.999710   
                     2021-05-04  0.285714  0.261905  1.023810  0.999710   
                     2021-07-02  0.285714  0.285714  1.023810  0.999710   
                     2021-12-02  0.275862  0.241379  0.827586  0.848064   
                     2021-12-03  0.300000  0.233333  0.833333  0.833908   

                                 FTG_rival_mean  FTG_rival_std  
idTeam Team     Side Date                                       
0      Aachen   0    2006-03-12             NaN            NaN  
                     2006-04-11             NaN            NaN  
                     2006-08-19             NaN            NaN  
                     2006-09-16             NaN            NaN  
                     2006-09-30             NaN            NaN  
...                                         ...            ...  
317    Zaragoza 1    2021-04-30        1.261905       0.989198  
                     2021-05-04        1.285714       0.994760  
                     2021-07-02        1.214286       0.976198  
                     2021-12-02        1.206897       0.861034  
                     2021-12-03        1.200000       0.846901  

[117092 rows x 6 columns]

In [65]:
aggs = {"Draw":["mean"],"Win":["mean"],"FTG":["mean","std"],"FTG_rival":["mean","std"]}
cols_aggs = ["Draw","Win","FTG_mean","FTG_std","FTG_rival_mean","FTG_rival_std"]
short_term = ut.compute_lag(data_split.sort_values("Date"),5,5,'Date',["idTeam","Team","season"],aggs,cols_aggs)
short_term

Draw  Win  FTG_mean   FTG_std   
idTeam Team     season Date                                        
0      Aachen   T07-08 2006-03-12   NaN  NaN       NaN       NaN  \
                       2006-04-11   NaN  NaN       NaN       NaN   
                       2006-07-11   NaN  NaN       NaN       NaN   
                       2006-08-19   NaN  NaN       NaN       NaN   
                       2006-08-26   NaN  NaN       NaN       NaN   
...                                 ...  ...       ...       ...   
317    Zaragoza T20-21 2021-08-01   0.4  0.6       1.2  0.836660   
                       2021-08-05   0.2  0.8       1.6  0.547723   
                       2021-11-04   0.2  0.8       1.2  0.836660   
                       2021-12-02   0.2  0.8       1.4  0.894427   
                       2021-12-03   0.4  0.6       1.4  0.894427   

                                   FTG_rival_mean  FTG_rival_std  
idTeam Team     season Date                                       
0      Aachen   T07-08 2006-03-12             NaN            NaN  
                       2006-04-11             NaN            NaN  
                       2006-07-11             NaN            NaN  
                       2006-08-19             NaN            NaN  
                       2006-08-26             NaN            NaN  
...                                           ...            ...  
317    Zaragoza T20-21 2021-08-01             0.6       0.894427  
                       2021-08-05             0.6       0.894427  
                       2021-11-04             0.2       0.447214  
                       2021-12-02             0.4       0.547723  
                       2021-12-03             0.6       0.547723  

[117092 rows x 6 columns]

In [66]:
feats = [ [f+s for f in cols_aggs] for s in ['_H','_A']  ]
feats = [*feats[0],*feats[1]]
feats

['Draw_H',
 'Win_H',
 'FTG_mean_H',
 'FTG_std_H',
 'FTG_rival_mean_H',
 'FTG_rival_std_H',
 'Draw_A',
 'Win_A',
 'FTG_mean_A',
 'FTG_std_A',
 'FTG_rival_mean_A',
 'FTG_rival_std_A']

In [67]:
data_split

,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,Loss,Draw,Win
6316,6316,D1,1993-01-09,T93-94,2,Bayern Munich,3.0,0.0,0,3,False,False,True
6316,6316,D1,1993-01-09,T93-94,25,Leipzig,0.0,3.0,1,0,True,False,False
6317,6317,D1,1993-01-09,T93-94,8,Dortmund,4.0,0.0,0,3,False,False,True
6317,6317,D1,1993-01-09,T93-94,9,Dresden,0.0,4.0,1,0,True,False,False
6318,6318,D1,1993-01-09,T93-94,12,Ein Frankfurt,3.0,1.0,0,3,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
44863,44863,SP1,2021-12-05,T20-21,200,Getafe,0,1,1,0,True,False,False
44864,44864,SP1,2021-12-05,T20-21,205,Huesca,1,0,0,3,False,False,True
44864,44864,SP1,2021-12-05,T20-21,188,Ath Bilbao,0,1,1,0,True,False,False
44865,44865,SP1,2021-12-05,T20-21,189,Ath Madrid,2,1,0,3,False,False,True


In [68]:
short_term = short_term.reset_index()
short_term

,idTeam,Team,season,Date,Draw,Win,FTG_mean,FTG_std,FTG_rival_mean,FTG_rival_std
0,0,Aachen,T07-08,2006-03-12,NaN,NaN,NaN,NaN,NaN,NaN
1,0,Aachen,T07-08,2006-04-11,NaN,NaN,NaN,NaN,NaN,NaN
2,0,Aachen,T07-08,2006-07-11,NaN,NaN,NaN,NaN,NaN,NaN
3,0,Aachen,T07-08,2006-08-19,NaN,NaN,NaN,NaN,NaN,NaN
4,0,Aachen,T07-08,2006-08-26,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
117087,317,Zaragoza,T20-21,2021-08-01,0.4,0.6,1.2,0.836660,0.6,0.894427
117088,317,Zaragoza,T20-21,2021-08-05,0.2,0.8,1.6,0.547723,0.6,0.894427
117089,317,Zaragoza,T20-21,2021-11-04,0.2,0.8,1.2,0.836660,0.2,0.447214
117090,317,Zaragoza,T20-21,2021-12-02,0.2,0.8,1.4,0.894427,0.4,0.547723


In [69]:
short_term = short_term.merge(data_split[['Date','idTeam','Side','matchId']],on=['Date','idTeam'])
short_term

,idTeam,Team,season,Date,Draw,Win,FTG_mean,FTG_std,FTG_rival_mean,FTG_rival_std,Side,matchId
0,0,Aachen,T07-08,2006-03-12,NaN,NaN,NaN,NaN,NaN,NaN,0,2147
1,0,Aachen,T07-08,2006-04-11,NaN,NaN,NaN,NaN,NaN,NaN,0,2096
2,0,Aachen,T07-08,2006-07-11,NaN,NaN,NaN,NaN,NaN,NaN,1,2104
3,0,Aachen,T07-08,2006-08-19,NaN,NaN,NaN,NaN,NaN,NaN,0,2024
4,0,Aachen,T07-08,2006-08-26,NaN,NaN,NaN,NaN,NaN,NaN,1,2035
...,...,...,...,...,...,...,...,...,...,...,...,...
117087,317,Zaragoza,T20-21,2021-08-01,0.4,0.6,1.2,0.836660,0.6,0.894427,0,56582
117088,317,Zaragoza,T20-21,2021-08-05,0.2,0.8,1.6,0.547723,0.6,0.894427,0,56771
117089,317,Zaragoza,T20-21,2021-11-04,0.2,0.8,1.2,0.836660,0.2,0.447214,0,56728
117090,317,Zaragoza,T20-21,2021-12-02,0.2,0.8,1.4,0.894427,0.4,0.547723,1,56625


In [70]:
def merge_sides(df,group_on,col_order):
    mask_home = df.Side==0
    mask_away = df.Side==1
    data_merged = df[mask_home].merge(df[mask_away], on=group_on, suffixes=("_H","_A"))
    data_merged = data_merged[col_order]
    return data_merged

In [75]:
merge_sides(short_term,["matchId"],col_order=["matchId",*feats])

,matchId,Draw_H,Win_H,FTG_mean_H,FTG_std_H,FTG_rival_mean_H,FTG_rival_std_H,Draw_A,Win_A,FTG_mean_A,FTG_std_A,FTG_rival_mean_A,FTG_rival_std_A
0,2147,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2096,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2042,0.2,0.2,1.4,1.341641,1.6,1.816590,0.2,0.2,0.4,0.547723,1.0,0.707107
4,2060,0.2,0.4,1.6,1.816590,1.0,1.000000,0.2,0.2,0.8,0.836660,1.2,0.447214
...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,56747,0.2,0.2,0.4,0.547723,1.2,1.095445,0.4,0.2,0.4,0.547723,0.6,0.894427
58542,56660,0.6,0.2,0.8,0.836660,1.2,1.303840,1.0,0.0,0.6,0.547723,0.6,0.547723
58543,56582,0.4,0.6,1.2,0.836660,0.6,0.894427,0.6,0.0,0.6,0.894427,1.6,1.516575
58544,56771,0.2,0.8,1.6,0.547723,0.6,0.894427,0.6,0.4,2.0,1.414214,0.6,0.547723


In [81]:
long_term = long_term.reset_index().merge(data_split[['Date','idTeam','matchId']],on=['Date','idTeam'])
long_term

,idTeam,Team,Side,Date,Draw,Win,FTG_mean,FTG_std,FTG_rival_mean,FTG_rival_std,matchId
0,0,Aachen,0,2006-03-12,NaN,NaN,NaN,NaN,NaN,NaN,2147
1,0,Aachen,0,2006-04-11,NaN,NaN,NaN,NaN,NaN,NaN,2096
2,0,Aachen,0,2006-08-19,NaN,NaN,NaN,NaN,NaN,NaN,2024
3,0,Aachen,0,2006-09-16,NaN,NaN,NaN,NaN,NaN,NaN,2042
4,0,Aachen,0,2006-09-30,NaN,NaN,NaN,NaN,NaN,NaN,2060
...,...,...,...,...,...,...,...,...,...,...,...
117087,317,Zaragoza,1,2021-04-30,0.261905,0.285714,1.023810,0.999710,1.261905,0.989198,56758
117088,317,Zaragoza,1,2021-05-04,0.285714,0.261905,1.023810,0.999710,1.285714,0.994760,56722
117089,317,Zaragoza,1,2021-07-02,0.285714,0.285714,1.023810,0.999710,1.214286,0.976198,56619
117090,317,Zaragoza,1,2021-12-02,0.275862,0.241379,0.827586,0.848064,1.206897,0.861034,56625


In [82]:
merge_sides(long_term,["matchId"],col_order=["matchId",*feats])

,matchId,Draw_H,Win_H,FTG_mean_H,FTG_std_H,FTG_rival_mean_H,FTG_rival_std_H,Draw_A,Win_A,FTG_mean_A,FTG_std_A,FTG_rival_mean_A,FTG_rival_std_A
0,2147,NaN,NaN,NaN,NaN,NaN,NaN,0.133333,0.200000,1.066667,0.961150,2.066667,1.387015
1,2096,NaN,NaN,NaN,NaN,NaN,NaN,0.388889,0.305556,1.000000,0.894427,1.138889,0.990030
2,2024,NaN,NaN,NaN,NaN,NaN,NaN,0.235294,0.411765,1.117647,1.007989,1.147059,1.158169
3,2042,NaN,NaN,NaN,NaN,NaN,NaN,0.290323,0.096774,0.741935,0.773207,1.677419,1.275071
4,2060,NaN,NaN,NaN,NaN,NaN,NaN,0.133333,0.200000,0.733333,0.798809,1.933333,1.334523
...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,56747,0.235294,0.411765,1.205882,1.066839,1.117647,1.225109,0.307692,0.256410,0.641026,0.742938,0.948718,0.971941
58542,56660,0.264706,0.382353,1.088235,0.965076,1.058824,1.229466,0.230769,0.230769,0.871795,1.173826,1.128205,0.893823
58543,56582,0.264706,0.411765,1.117647,0.945955,1.000000,1.230915,0.411765,0.176471,0.647059,0.861770,1.294118,1.311712
58544,56771,0.257143,0.428571,1.142857,0.943799,0.971429,1.224402,0.333333,0.444444,1.444444,1.199128,0.777778,0.646762


# ENSAMBLE

In [15]:
import dataflow_pi_rating as pirates

def get_labels(df,col,new_col):
    df.loc[:,new_col] = df.loc[:,col]
    return ut._create_label(df,new_col)

#### end